In [1]:
# ========== 第 1 步 — 导入：把后面要用的工具箱搬进来 ==========

# 导入 requests：用 HTTP 抓取网页 HTML
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的 DOM 树
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮渲染摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：本练习用它走 Ollama 的 OpenAI 兼容接口
from openai import OpenAI


In [2]:
# ========== 第 2 步 — 连接本地 Ollama（OpenAI 兼容 API）==========

# Ollama 默认在 localhost:11434 暴露 /v1 兼容接口；模型需事先 ollama pull
# api_key 是 OpenAI SDK 的必填字段；本地 Ollama 会忽略它，随便填占位即可

# 创建指向本地 Ollama 的客户端：base_url 指向兼容层 /v1
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# 模型名字符串必须和本机已安装的模型一致（例如 llama3.2）
MODEL = "llama3.2"


In [3]:
# ========== 第 3 步 — 快速连通性测试：确认 Ollama + 模型能应答 ==========

# 发一条极短的 user 消息；不设 system，只测链路是否通
response = ollama.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello\! Reply with one sentence only."}]
)
# choices[0].message.content：取第一条候选回复的正文并打印
print(response.choices[0].message.content)


<>:5: SyntaxWarning: invalid escape sequence '\!'
<>:5: SyntaxWarning: invalid escape sequence '\!'
/var/folders/32/fdngh_l52yd4cpmdpm6c4_000000gn/T/ipykernel_28551/2935225330.py:5: SyntaxWarning: invalid escape sequence '\!'
  messages=[{"role": "user", "content": "Hello\! Reply with one sentence only."}]


I'm happy to assist you.


In [4]:
# ========== 第 4 步 — 网站抓取辅助函数：URL → 清洗后的纯文本 ==========

# 伪装成常见浏览器 User-Agent，降低被目标站直接拒绝的概率
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    """抓取任意 URL，去掉脚本/样式等噪声，返回 title + 正文（截断到 2000 字符）。"""
    # GET 目标页；带上 HEADERS，避免部分站点对默认 UA 返回异常页
    response = requests.get(url, headers=HEADERS)
    # 用 html.parser 解析响应字节为 BeautifulSoup 文档树
    soup = BeautifulSoup(response.content, "html.parser")
    # 优先取 <title>；没有则给占位文案（字符串本身保留英文，不影响逻辑）
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        # 删除 body 内无关标签：脚本、样式、图片、表单输入（不进摘要）
        for tag in soup.body(["script", "style", "img", "input"]):
            tag.decompose()
        # 抽出纯文本：换行分隔、去掉首尾空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 无 body 时正文为空串
        text = ""
    # 标题与正文用空行拼接，再截断到 2000 字符，控制送进模型的上下文长度
    return (title + "\n\n" + text)[:2_000]


In [5]:
# ========== 第 5 步 — 试用抓取器：先看清洗结果长什么样 ==========

# 抓取课程作者站点；可改成任意公开 URL 做试验
content = fetch_website_contents("https://edwarddonner.com")
# 打印原始文本，确认标题与正文是否合理、噪声是否已去掉
print(content)


Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190

In [6]:
# ========== 第 6 步 — 定义 system / user 提示词（发给模型的英文指令勿译）==========

# system_prompt：定调「怎么答」——毒舌、幽默、短摘要、Markdown，忽略导航文案
system_prompt = """
You are a snarky assistant that analyzes the contents of a website
and provides a short, snarky, humorous summary, ignoring navigation-related text.
Respond in markdown. Do not wrap the markdown in a code block.
"""

# user_prompt_prefix：user 消息的固定前缀；后面会拼接网页正文
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.

"""


In [7]:
# ========== 第 7 步 — 构建 messages 列表（Chat Completions 标准格式）==========

# OpenAI 兼容格式：system 定角色，user 放具体网页内容

def messages_for(website_content):
    """把网页文本包装成 [system, user] 两条消息，供 chat.completions.create 使用。"""
    return [
        # system：全局行为约束
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 实际网页正文
        {"role": "user", "content": user_prompt_prefix + website_content}
    ]

# 用假正文预览消息结构（不调 API，只看 list/dict 长什么样）
messages_for("Example website content here")


[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website\nand provides a short, snarky, humorous summary, ignoring navigation-related text.\nRespond in markdown. Do not wrap the markdown in a code block.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, summarize these too.\n\nExample website content here'}]

In [8]:
# ========== 第 8 步 — summarize()：抓取 URL → 调 Ollama → 返回摘要文本 ==========

def summarize(url):
    """端到端：抓网页 → 组 messages → 调用本地模型 → 返回摘要字符串。"""
    # 先抓取并清洗目标站正文
    website = fetch_website_contents(url)
    # 非流式 Chat Completions：等整段生成完再返回
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    # 取出模型回复正文
    return response.choices[0].message.content


In [10]:
# ========== 第 9 步 — display_summary()：把摘要渲染成笔记本里的 Markdown ==========

def display_summary(url):
    """调用 summarize，再用 IPython display + Markdown 漂亮展示。"""
    # 拿到模型生成的 Markdown 字符串
    summary = summarize(url)
    # 在笔记本输出区渲染（不是纯 print 纯文本）
    display(Markdown(summary))


In [11]:
# ========== 第 10 步 — 运行！对 CNN 首页做毒舌摘要 ==========

# 第一次真实端到端调用：抓取 + 推理可能较慢，属正常
display_summary("https://cnn.com")


**Summary**: CNN's website is a news hub with various sections, including Breaking News, Politics, Business, Entertainment, and more. It also offers videos, live TV, and podcasts. Yay, another online news site.

**News and Announcements**

* The website asks for user feedback on ads, which can be submitted via a handy form. Because who doesn't love giving constructive criticism?
* CNN has made it clear that *everything* on the site is "Breaking News," which kind of defeats the purpose.
* A dedicated section exists for submitting feedback on ads, making it easy to let your thoughts be heard (or annoy others with them).
* Various sub-sections cover topics like politics, business, and health, but they're all just "News" in their respective categories. You'd think there would be more depth or categorization, but nope.
* Unfortunately, the website asks you to log in for even basic functions, which is a real bummer after spending all that time on the ad feedback form.

In [12]:
# ========== 再试 Anthropic 官网：对比不同站点摘要风格 ==========

display_summary("https://anthropic.com")


This website appears to be about Anthropic, a public benefit corporation that aims to secure the benefits and mitigate the risks of AI. They've created an AI platform called Claude, which offers a space for genuine conversations without ads or sponsored content.

News:

* 81,000 people participated in the largest study ever done on AI, and its findings are supposedly summarized somewhere on this site (but not found among these contents).
* Anthropic released version 4.6 of their AI model, Claude Opus, on February 5, 2026.
* There's a recent announcement or two about new releases of the Claude platform, but I couldn't dig up the content since it was buried under annoying navigation menus and other non-essential text.

That's basically it – more of a corporate update than actual news.

In [13]:
# ========== 再试 edwarddonner.com：个人站 / 课程相关内容 ==========

display_summary("https://edwarddonner.com")


Ed's website is basically an intro to his life, interests, and work. He's a guy who likes writing code, playing with large language models (LLMs), making music, and... also making more code, because why not? On the 'About' page, he talks about co-founding Nebula.io, his previous start-up untapt getting acquired in 2021, and how much joy it brings him to share LLM info with anyone who'll listen (he even convinced others to make some Udemy courses which did surprisingly well).

He also shares some recent updates like new resources on AI development and sharing them as news-like entries.